### Listing 8.3: Setting up the Benchmark and Target Model

In [1]:
import torch
import re
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

benchmark_sample = {
    "question": "What is the capital of Australia?",
    "expected": "Canberra is the capital city of Australia."
}

TARGET_MODEL = "google/gemma-3-270m-it"
print(f"Loading Target Model: {TARGET_MODEL}...")

target_tok = AutoTokenizer.from_pretrained(TARGET_MODEL)
target_mdl = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)

Loading Target Model: google/gemma-3-270m-it...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

### Listing 8.4: Crafting the Generation Function

In [2]:
def generate_answer(question: str) -> str:
    """Generates an answer using the target small language model."""
    messages = [
        {"role": "user", "content": f"Answer briefly and factually: {question}"}
    ]
    
    prompt = target_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = target_tok(prompt, return_tensors="pt").to(target_mdl.device)
    
    with torch.no_grad():
        outputs = target_mdl.generate(
            **inputs, 
            max_new_tokens=50,
            do_sample=True,
            temperature=0.1
        )
        
    generated_text = target_tok.decode(
        outputs[0][inputs.input_ids.shape[-1]:], 
        skip_special_tokens=True
    ).strip()
    
    return generated_text

print("Generating answer...")
generated_answer = generate_answer(benchmark_sample["question"])
print(f"Target Output: {generated_answer}\n")

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Generating answer...
Target Output: The capital of Australia is Canberra.



### Listing 8.5:  Loading the Judge model

In [3]:
print("Unloading Target Model to free VRAM...")
del target_mdl
del target_tok
gc.collect()
torch.cuda.empty_cache()

JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
print(f"Loading Judge Model: {JUDGE_MODEL}...")

judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, 
    device_map="auto"
)

Unloading Target Model to free VRAM...
Loading Judge Model: Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### Listing 8.6: Setting the Evaluation Logic

In [4]:
def ai_judge(question: str, generated: str, expected: str) -> float:
    """Uses a larger judge model to score factuality from 1 to 10."""
    prompt = (
        "Evaluate the factuality of the generated answer against the reference.\n"
        f"Question: {question}\n"
        f"Reference: {expected}\n"
        f"Generated: {generated}\n\n"
        "First write your reasoning. Then, on the very last line, write ONLY an integer score from 1 to 10."
    )

    inp = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        return_tensors="pt",
        add_generation_prompt=True
    ).to(judge_mdl.device)

    with torch.no_grad():
        out = judge_mdl.generate(
            **inp,
            max_new_tokens=250,
            do_sample=False
        )

    txt = judge_tok.decode(out[0, inp['input_ids'].shape[-1] :], skip_special_tokens=True).strip()
    print(f"--- Judge Reasoning ---\n{txt}\n-----------------------")
    m = re.search(r"\b(\d+)\b", txt.splitlines()[-1])
    return int(m.group(1)) if m else 0

### Listing 8.7: Executing the Evaluation

In [5]:
print("Evaluating the target model's answer...")
score = ai_judge(
    question=benchmark_sample["question"],
    generated=generated_answer,
    expected=benchmark_sample["expected"]
)

print(f"\nFinal Factuality Score: {score}")

Evaluating the target model's answer...
--- Judge Reasoning ---
Let's break down the evaluation:

1. **Question**: The question asks for the capital of Australia.
2. **Reference Answer**: The reference states that Canberra is the capital city of Australia.
3. **Generated Answer**: The generated answer states that Canberra is the capital of Australia.

The reference and the generated answer both provide the same information about Canberra being the capital of Australia. There is no discrepancy between the two statements.

**Reasoning**:
- Both the reference and the generated answer correctly identify Canberra as the capital of Australia.
- The wording in the reference ("capital city") matches the wording in the generated answer ("capital").
- No additional or contradictory information is provided in either statement.

Given this analysis, the generated answer is factually correct and aligns with the reference.

**Score**: 10
-----------------------

Final Factuality Score: 10
